# Paper figure notebook

Regenerates the figures of the main text from the snapshot under `data/`.
Each plotting cell loads its own inputs and saves PNG and PDF files to `output/`.

| Output | Figure | Input |
|---|---|---|
| `fig1_cliff_boundary` | Figure 1 (right): adaptive cliff threshold | computed inline |
| `fig2_avg_cliff_tokens` | Figure 2: average cliff tokens per trace | `data/fig2_avg_cliff_tokens/rates.json` |
| `fig3_passk_incorrect` | Figure 3: Cliff-del vs. Cliff-keep pass@k, incorrect traces | `data/fig3_passk_incorrect/pass_at_k.csv` |
| `fig4_self_correction` | Figure 4: success probability across a self-correcting trace | `data/fig4_self_correction/trace.json` |

Snapshot sources:

| Snapshot | Produced by |
|---|---|
| `fig2_avg_cliff_tokens/rates.json` | `scripts/run_exp1_occurrence.sh` → `output/04_cliff_occurrence/<run>/avg_cliff_tokens.json` |
| `fig3_passk_incorrect/pass_at_k.csv` | `scripts/run_exp1_deletion.sh` → `output/05_deletion_ablation/<run>/pass_at_k_first_cliff_incorrect.csv` |
| `fig4_self_correction/trace.json` | `scripts/build_self_correction_case.py` |

Run all cells from top to bottom.


In [ ]:
import csv, json, math
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT / "figure/data").exists():
    ROOT = ROOT / "figure"
DATA = ROOT / "data"
OUT = ROOT / "output"; OUT.mkdir(exist_ok=True)

def save_fig(fig, name, dpi=170):
    fig.savefig(OUT / f"{name}.png", dpi=dpi, bbox_inches="tight")
    fig.savefig(OUT / f"{name}.pdf", format="pdf", bbox_inches="tight")
    print(f"saved {name}.png + .pdf")

# Shared style constants
SUCCESS_COLOR = "#90CAF9"
FAILURE_COLOR = "#EF9A9A"

def apply_style():
    plt.rcParams.update({
        "figure.dpi": 150, "font.size": 14,
        "axes.titlesize": 16, "axes.labelsize": 15, "legend.fontsize": 13,
        "xtick.labelsize": 13, "ytick.labelsize": 13,
    })
    sns.set_style("whitegrid")

# Display names for models.
MODEL_DISPLAY = {
    "Llama-3.1-8B-Instruct": "Llama-3.1-8B",
    "Llama-3.2-3B-Instruct": "Llama-3.2-3B",
    "Llama-3.2-1B-Instruct": "Llama-3.2-1B",
    "gemma-3-4b-it":         "Gemma-3-4B",
}
def display_model(m):
    return MODEL_DISPLAY.get(m, m)

def base_dataset(name):
    """Pipeline subsets are named gsm1k_100 / math500_100."""
    return name[:-len("_100")] if name.endswith("_100") else name


In [ ]:
# ─── Figure 1 (right): adaptive cliff threshold (no input data) ───
# Constants from src/analysis/cliff_threshold.py:
#   DEFAULT_N=64, DEFAULT_DELTA_0=0.1, DEFAULT_Z_ALPHA=1.645
N, delta_0, z_alpha = 64, 0.1, 1.645

def min_drop_continuous(p_prev):
    """For a given p_prev, find the minimum drop that triggers cliff."""
    if p_prev <= 0:
        return None
    lo, hi = delta_0, p_prev
    if lo >= hi:
        return None
    for _ in range(100):
        mid = (lo + hi) / 2
        p_curr = max(p_prev - mid, 0)
        var = p_prev * (1 - p_prev) / N + p_curr * (1 - p_curr) / N
        se = math.sqrt(var) if var > 0 else 0
        threshold = delta_0 + z_alpha * se
        if mid > threshold:
            hi = mid
        else:
            lo = mid
    return (lo + hi) / 2

x_pts, y_pts = [], []
for p in np.linspace(0.001, 1.0, 1000):
    d = min_drop_continuous(p)
    if d is not None and d < p - 1e-6:
        x_pts.append(p); y_pts.append(d)
x_boundary, y_boundary = np.array(x_pts), np.array(y_pts)
# prepend the point where boundary meets the diagonal so the line starts
# cleanly from the diagonal
if len(x_boundary) > 0:
    x0 = x_boundary[0]
    x_boundary = np.concatenate([[x0 - 0.001], x_boundary])
    y_boundary = np.concatenate([[x0 - 0.001], y_boundary])

YMAX = 1.0
fig, ax = plt.subplots(figsize=(6, 6))
ax.grid(True, color="#e0e0e0", linewidth=0.6); ax.set_axisbelow(True)

# Set x/y axis spines to 1pt
for spine in ax.spines.values():
    spine.set_linewidth(1.0)

diag_x = np.linspace(0, 1.0, 500)
ax.fill_between(diag_x, diag_x, YMAX, color="#000000", alpha=0.55,
                label="Infeasible region")
upper = np.minimum(x_boundary, YMAX)
ax.fill_between(x_boundary, y_boundary, upper, color="#FFD4DB", alpha=0.45,
                label="Cliff token region", zorder=2)
ax.plot(x_boundary, y_boundary, color="#EA4025", linewidth=2.2,
        label="Adaptive threshold", zorder=3)

ax.set_xlabel(r"Pre-cliff success probability  ($p_{t-1}$)", fontsize=13)
ax.set_ylabel("Success probability drop ($p_{t-1} - p_{t}$)", fontsize=13)
ax.set_xlim(0, 1.0); ax.set_ylim(0, YMAX)
ax.set_xticks(np.arange(0, 1.1, 0.1))
ax.set_yticks(np.arange(0, 1.1, 0.1))
ax.set_aspect("equal")
ax.legend(loc="upper left", fontsize=15, framealpha=1, borderpad=0.5)
fig.tight_layout()
save_fig(fig, "fig1_cliff_boundary")
plt.show()
plt.close(fig)

In [ ]:
# ─── Figure 2: average cliff tokens per trace ───
from matplotlib.transforms import ScaledTranslation

rates_avg = json.loads((DATA / "fig2_avg_cliff_tokens/rates.json").read_text())

FS = {"tick": 16, "label": 20, "legend": 16, "annot": 16}
apply_style()

models = list(rates_avg.keys())
x = np.arange(len(models))
bar_w = 0.35

# Compact height
fig, ax = plt.subplots(figsize=(12.5, 4.2))

# Annotation offsets: blue to the left, red to the right
blue_label_shift = ScaledTranslation(-4 / 72, 0, fig.dpi_scale_trans)
red_label_shift = ScaledTranslation(4 / 72, 0, fig.dpi_scale_trans)

s_avg = [rates_avg[m]["success"] for m in models]
f_avg = [rates_avg[m]["failure"] for m in models]

# Bar outlines
bars_s = ax.bar(
    x - bar_w / 2, s_avg, bar_w,
    color=SUCCESS_COLOR,
    edgecolor="#1565C0",
    linewidth=1.6,
    label="Correct trace",
    zorder=3,
)
bars_f = ax.bar(
    x + bar_w / 2, f_avg, bar_w,
    color=FAILURE_COLOR,
    edgecolor="#C62828",
    linewidth=1.6,
    label="Incorrect trace",
    zorder=3,
)

# Blue labels, shifted left
for bar in bars_s:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{bar.get_height():.2f}",
        ha="center",
        va="bottom",
        fontsize=FS["annot"],
        color="#1565C0",
        fontweight="semibold",
        transform=ax.transData + blue_label_shift,
    )

# Red labels, shifted right
for bar in bars_f:
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.02,
        f"{bar.get_height():.2f}",
        ha="center",
        va="bottom",
        fontsize=FS["annot"],
        color="#C62828",
        fontweight="semibold",
        transform=ax.transData + red_label_shift,
    )

ax.set_xticks(x)
ax.set_xticklabels(
    [display_model(m) for m in models],
    fontsize=FS["tick"],
    fontweight="medium",
)
ax.set_ylabel(
    "Avg cliff tokens per trace",
    fontsize=FS["label"],
    fontweight="medium",
)
ax.set_ylim(0, max(f_avg) * 1.20)
ax.grid(False, axis="x")  # no vertical grid lines
ax.grid(True, axis="y", color="#E5E7EB", linewidth=0.65)  # keep horizontal grid lines
ax.legend(fontsize=FS["legend"], loc="upper right")

# Hide top/right spines
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# Black, thicker x/y axis lines
ax.spines["bottom"].set_color("black")
ax.spines["left"].set_color("black")
ax.spines["bottom"].set_linewidth(1.5)
ax.spines["left"].set_linewidth(1.5)

# Black tick marks
ax.tick_params(axis="both", colors="black", width=1.2)

fig.tight_layout()
save_fig(fig, "fig2_avg_cliff_tokens")
plt.show()
plt.close(fig)

In [ ]:
# ─── Figure 3: Cliff-del vs. Cliff-keep pass@k on incorrect traces (first cliff per trace) ───
import pandas as pd

df = pd.read_csv(DATA / "fig3_passk_incorrect/pass_at_k.csv")
df["dataset"] = df["dataset"].map(base_dataset)

MODEL_DISPLAY_ORDER = [
    "Qwen3-8B", "Qwen3-4B", "Qwen3-0.6B",
    "Llama-3.1-8B-Instruct", "Llama-3.2-3B-Instruct",
    "Llama-3.2-1B-Instruct", "gemma-3-4b-it",
]
DATASET_DISPLAY_ORDER = ["gsm1k", "math500", "aime25"]
DATASET_LABEL = {
    "gsm1k": "GSM1K",
    "math500": "MATH500",
    "aime25": "AIME 2025",
}

FS = {
    "tick": 24,
    "k_tick": 26,   # k-axis tick labels
    "title": 32,    # model names
    "label": 29,
    "xlabel": 32,   # x-axis label "k"
    "annot": 28,    # "n = ..." annotation
    "legend": 28,
}

TRACE_LINES = {
    "cliff_del":  {"color": "#5959DD", "marker": "o", "label": "Cliff-del"},
    "cliff_keep": {"color": "#E95C7E", "marker": "s", "label": "Cliff-keep"},
}

K_TICKS = [1, 2, 4, 8, 16, 32, 64]
Y_TICKS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]

apply_style()

n_rows, n_cols = len(DATASET_DISPLAY_ORDER), len(MODEL_DISPLAY_ORDER)
fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4 * n_cols, 3 * n_rows),
    squeeze=False,
)

def style_passk_axes(ax):
    ax.set_xscale("log", base=2)
    ax.set_xticks(K_TICKS)
    ax.set_xticklabels([str(k) for k in K_TICKS], fontsize=FS["tick"])
    ax.set_yticks(Y_TICKS)
    ax.set_yticklabels([f"{y:.1f}" for y in Y_TICKS], fontsize=FS["tick"])
    ax.set_xlim(K_TICKS[0] * 0.85, K_TICKS[-1] * 1.18)
    ax.set_ylim(-0.02, 1.02)

    # Grid
    ax.grid(
        True,
        which="major",
        color="#CCCCCC",
        linewidth=0.6,
        alpha=0.7,
        zorder=0,
    )
    ax.grid(True, axis="y", color="#CCCCCC", linewidth=0.6, alpha=0.7, zorder=0)
    ax.set_axisbelow(True)

    # Uniform dark-grey panel borders
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color("#555555")
        spine.set_linewidth(1.3)

    ax.tick_params(axis="both", colors="black", width=1.2)

for i, dataset in enumerate(DATASET_DISPLAY_ORDER):
    for j, model in enumerate(MODEL_DISPLAY_ORDER):
        ax = axes[i][j]
        style_passk_axes(ax)

        cell = df[
            (df["model"] == model) &
            (df["dataset"] == dataset)
        ].sort_values("k")

        n_traces = int(cell["n_traces"].iloc[0]) if not cell.empty else 0

        if cell.empty or n_traces == 0:
            ax.set_facecolor("#E0E0E0")
            ax.text(
                0.5, 0.5, "No cliff traces",
                transform=ax.transAxes,
                fontsize=FS["annot"],
                ha="center", va="center", color="#555555",
                in_layout=False,
            )
            ax.text(
                0.98, 0.02, "n = 0",
                transform=ax.transAxes,
                fontsize=FS["annot"],
                ha="right",
                va="bottom",
                color="black",
                in_layout=False,
            )
        else:
            for column, style in TRACE_LINES.items():

                ax.plot(
                    cell["k"],
                    cell[column],
                    color=style["color"],
                    linestyle="-",
                    linewidth=2.8,
                    marker=style["marker"],
                    markersize=11,
                    zorder=3,
                )

            ax.text(
                0.98, 0.02, f"n = {n_traces}",
                transform=ax.transAxes,
                fontsize=FS["annot"],
                ha="right",
                va="bottom",
                color="black",
                in_layout=False,
            )

        if i == 0:
            ax.set_title(
                display_model(model),
                fontsize=FS["title"],
                fontweight="medium",
            )

        if j == 0:
            ax.set_ylabel(
                DATASET_LABEL[dataset],
                fontsize=FS["label"],
                fontweight="medium",
            )

        if i == n_rows - 1:
            ax.set_xlabel("k", fontsize=FS["xlabel"], fontweight="medium")
        else:
            ax.tick_params(labelbottom=False)

        if j != 0:
            ax.tick_params(labelleft=False)

# Legend
handles = [
    Line2D(
        [0], [0],
        color=style["color"],
        marker=style["marker"],
        linestyle="-",
        linewidth=2.8,
        markersize=12,
        label=style["label"],
    )
    for style in TRACE_LINES.values()
]

fig.legend(
    handles=handles,
    loc="lower center",
    ncol=2,
    bbox_to_anchor=(0.5, 0.95),
    frameon=True,
    fontsize=FS["legend"],
)

fig.tight_layout(rect=[0, 0.04, 1, 0.97])
save_fig(fig, "fig3_passk_incorrect")
plt.show()
plt.close(fig)

In [ ]:
# ─── Figure 4: success probability across a self-correcting trace ───
# Case study of Section 4.3: Qwen3-4B, AIME 2025 Problem 1, a *correct* trace that
# still contains cliff tokens.  Top panel = estimated p(success) per token
# position; bottom = the same response as flowing text, every token shaded by
# that probability.  Boxed spans are the two statistical cliff tokens, the
# miscomputed value, and the self-correction cue.
#
# Input is `data/fig4_self_correction/trace.json`, the snapshot written by
# `scripts/build_self_correction_case.py` (which is where the cliff detector
# runs).  The text-layout engine is too long for a cell, so it stays in
# `figure/_fig_self_correction.py`.  It also sets its own typography (STIXGeneral +
# mathtext, 7pt at 7in wide) and saves both PNG and PDF itself, so unlike the
# other cells it does not go through `apply_style()` / `save_fig()`.
import sys

from IPython.display import Image, display

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from _fig_self_correction import load_snapshot, render

# Callout wording lives here rather than in the snapshot, so it can be kept in
# step with the paper text without rebuilding the trace.  One entry per mark,
# in snapshot order: cliff 1, the wrong value, cliff 2, the recovery cue.
# Cliff 2 (0.44→0.16) and the rebound at "But wait" (0.38→0.80) are the numbers
# quoted in Section 4.3.
FIG4_LABELS = [
    "cliff token 1  0.69→0.34",
    "wrong value 443 (should be 448)",
    "cliff token 2  0.44→0.16",
    'self-correction cue "But wait"  0.38→0.80',
]

fig4_snapshot = load_snapshot(DATA / "fig4_self_correction" / "trace.json")
fig4_info = render(
    fig4_snapshot, OUT,
    labels=FIG4_LABELS,
    show_callouts=True,    # leader line + label under every boxed span
    show_curve=True,       # p(success) panel on top
    show_colorbar=True,    # colour key for the text shading, bottom right
)
print(fig4_info)
display(Image(filename=str(OUT / "fig4_self_correction.png")))

